# Avocado Sales & Produce Market Analysis

This notebook analyzes weekly Hass avocado retail sales data from **2015–2018**, originally published by the **Hass Avocado Board**.

> Exact historical numeric results were not recoverable, so this notebook reconstructs the workflow without inventing outputs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


## Load the data

**Original data source:** Hass Avocado Board retail volume and price data.

Upload the original or equivalent CSV and name it avocado.csv.

In [ ]:
df = pd.read_csv('avocado.csv')
df.head()


## Clean and prepare the data

In [ ]:
df.columns = [c.strip() for c in df.columns]

if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month

if 'type' in df.columns:
    df['type'] = df['type'].astype(str).str.strip().str.lower()

df = df.drop_duplicates()
print(df.shape)
df.head()


## Average price distribution

In [ ]:
if 'AveragePrice' in df.columns:
    plt.figure(figsize=(9,5))
    plt.hist(df['AveragePrice'].dropna(), bins=30)
    plt.title('Distribution of Average Avocado Price')
    plt.xlabel('Average Price')
    plt.ylabel('Frequency')
    plt.show()


## Organic vs. conventional pricing

In [ ]:
if {'type','AveragePrice'}.issubset(df.columns):
    print(df.groupby('type')['AveragePrice'].agg(['count','mean','median','std']))
    sns.boxplot(data=df, x='type', y='AveragePrice')
    plt.title('Average Price by Avocado Type')
    plt.show()


## Regional sales volume

In [ ]:
volume_col = next((c for c in ['Total Volume','TotalVolume','total_volume'] if c in df.columns), None)
if volume_col and 'region' in df.columns:
    regional = df.groupby('region')[volume_col].sum().sort_values(ascending=False).head(15)
    regional.sort_values().plot(kind='barh', figsize=(10,6))
    plt.title('Top Regions by Total Avocado Sales Volume')
    plt.show()


## Price trends over time

In [ ]:
if {'Date','AveragePrice'}.issubset(df.columns):
    trend = df.groupby('Date', as_index=False)['AveragePrice'].mean()
    plt.figure(figsize=(11,5))
    plt.plot(trend['Date'], trend['AveragePrice'])
    plt.title('Average Avocado Price Over Time')
    plt.show()


## Confidence interval and hypothesis testing

In [ ]:
if 'AveragePrice' in df.columns:
    x = df['AveragePrice'].dropna()
    ci = stats.t.interval(0.95, df=len(x)-1, loc=x.mean(), scale=stats.sem(x))
    print('95% CI for mean AveragePrice:', ci)

if {'type','AveragePrice'}.issubset(df.columns):
    organic = df.loc[df['type']=='organic','AveragePrice'].dropna()
    conventional = df.loc[df['type']=='conventional','AveragePrice'].dropna()
    print(stats.ttest_ind(organic, conventional, equal_var=False))


## Chi-square test: region × type

In [ ]:
if {'region','type'}.issubset(df.columns):
    table = pd.crosstab(df['region'], df['type'])
    chi2, p, dof, expected = stats.chi2_contingency(table)
    print('Chi-square:', chi2)
    print('p-value:', p)


## Findings

Add final conclusions only after the notebook is run against the Hass Avocado Board dataset.